In [1]:
# import current working diretories

import os
import sys
print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\research


In [2]:
# system path

sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction")

In [3]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction\\research'

In [4]:
# changing to the parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\Hotel_Booking_Cancellation_Prediction'

In [6]:
# import box versions

import box
print(box.__version__)

7.4.1


In [7]:
# entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:

    root_dir: Path
    model_path: Path
    test_data_path: Path
    metric_file_name: Path

In [8]:
from Hotel_Booking_Cancellation_Prediction.constant import *
from Hotel_Booking_Cancellation_Prediction.utils.common import read_yaml, create_directories
from Hotel_Booking_Cancellation_Prediction.entity.config_entity import ModelEvaluationConfig
from Hotel_Booking_Cancellation_Prediction.config.configuration import ConfigurationManager


In [9]:
# configuration manager

def get_model_evaluation_config(self) -> ModelEvaluationConfig:

    config = self.config["model_evaluation"]

    create_directories([config.root_dir])

    model_evaluation_config = ModelEvaluationConfig(

        root_dir=Path(config.root_dir),
        model_path=Path(config.model_path),
        test_data_path=Path(config.test_data_path),
        metric_file_name=Path(config.metric_file_name)
    )

    return model_evaluation_config

In [10]:
import pandas as pd
import joblib
import numpy as np
from Hotel_Booking_Cancellation_Prediction.logging import logger

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss
)


In [11]:
# components



class ModelEvaluation:

    def __init__(self, config):

        self.config = config

    # LOAD MODEL

    def load_model(self):

        model = joblib.load(self.config.model_path)

        return model

    # LOAD TEST DATA

    def load_data(self):

        test_df = pd.read_csv(self.config.test_data_path)

        return test_df

    # EVALUATE MODEL

    def evaluate_model(self):

        # Load model

        model = self.load_model()

        # Load test data

        test_df = self.load_data()

        # Separate features and target

        X_test = test_df.drop(columns=["is_canceled"])

        y_test = test_df["is_canceled"]

        # Predictions

        y_pred = model.predict(X_test)

        # Probability / decision scores

        if hasattr(model,"predict_proba"):

            y_prob = model.predict_proba(X_test)[:, 1]

        else:

            y_score = model.decision_function(X_test)

            y_prob = (
                1 /
                (
                    1 +
                    __import__("numpy").exp(
                        -y_score
                    )
                )
            )

        # Confusion Matrix

        tn, fp, fn, tp = confusion_matrix(y_test,y_pred
        ).ravel()

        # Accuracy

        accuracy = accuracy_score(y_test,y_pred)

        # Precision

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        # Recall

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        # Specificity

        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        # F1 Score

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        # ROC-AUC

        roc_auc = roc_auc_score(
            y_test,
            y_prob
        )

        # PR-AUC

        pr_auc = average_precision_score(
            y_test,
            y_prob
        )

        # Log Loss

        logloss = log_loss(
            y_test,
            y_prob
        )

        # Create results

        results = {

            "Accuracy": accuracy,

            "Precision": precision,

            "Recall": recall,

            "Specificity": specificity,

            "F1 Score": f1,

            "ROC-AUC": roc_auc,

            "PR-AUC": pr_auc,

            "Log Loss": logloss
        }

        # Display results

        print("\nFinal Model Evaluation:")

        for metric, value in results.items():

            print(f"{metric}: {value:.4f}")

        # Save results

        results_df = pd.DataFrame([results])

        results_df.to_csv(
            self.config.metric_file_name,
            index=False
        )

        print( "\nEvaluation metrics saved at:")

        print(self.config.metric_file_name)

        return results_df

In [12]:
# pipeline

class ModelEvaluationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        try:

            logger.info(">>>>>> Model Evaluation Stage Started <<<<<<")

            config = ConfigurationManager()

            model_evaluation_config = (config.get_model_evaluation_config() )

            model_evaluation = ModelEvaluation(config=model_evaluation_config)

            model_evaluation.evaluate_model()

            logger.info(">>>>>> Model Evaluation Stage Completed <<<<<<")

        except Exception as e:

            logger.exception(e)
            raise e

In [13]:
obj = ModelEvaluationTrainingPipeline()
obj.main()

[2026-08-17 17:52:04,393: INFO: 1438925037: >>>>>> Model Evaluation Stage Started <<<<<<]
[2026-08-17 17:52:04,407: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\config\config.yaml loaded successfully]
[2026-08-17 17:52:04,417: INFO: common: yaml file: C:\Users\Greesha Vaishnavi\Desktop\dsprojects\Hotel_Booking_Cancellation_Prediction\params.yaml loaded successfully]
[2026-08-17 17:52:04,420: INFO: common: created directory at artifacts]
[2026-08-17 17:52:04,422: INFO: common: created directory at artifacts/model_evaluation]

Final Model Evaluation:
Accuracy: 0.8923
Precision: 0.8698
Recall: 0.8341
Specificity: 0.9266
F1 Score: 0.8516
ROC-AUC: 0.9596
PR-AUC: 0.9409
Log Loss: 0.2798

Evaluation metrics saved at:
artifacts\model_evaluation\metrics.csv
[2026-08-17 17:52:08,662: INFO: 1438925037: >>>>>> Model Evaluation Stage Completed <<<<<<]
